# CH06 -- Notebook thực hành
Chạy lần lượt từng ô từ trên xuống. Đặt notebook ở thư mục gốc dự án (cùng cấp với thư mục `data/` và `figures/`).

### Ô lệnh 1 -- Nạp thư viện, dữ liệu và biến đổi đặc trưng cho mạng nơ-ron

In [ ]:
import sys
sys.path.append("code")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import QuantileTransformer
from common import credit_split

pd.set_option("display.width", 80)
pd.set_option("display.precision", 3)
SEED = 42
torch.set_num_threads(2)

X_train, X_test, y_train, y_test = credit_split()
qt = QuantileTransformer(output_distribution="normal", random_state=0).fit(X_train)
Xtr, Xte = qt.transform(X_train), qt.transform(X_test)      # mảng NumPy đã biến đổi
print("PyTorch:", torch.__version__, "| Xtr:", Xtr.shape, "| Xte:", Xte.shape)
print("BILL_AMT1 trước biến đổi: min =", X_train["BILL_AMT1"].min(), ", max =", X_train["BILL_AMT1"].max())
print("BILL_AMT1 sau biến đổi  : min = %.2f, max = %.2f" % (Xtr[:, 11].min(), Xtr[:, 11].max()))


### Ô lệnh 2 -- Tính tay focal loss cho 4 khách hàng

In [ ]:
y = np.array([1, 1, 0, 0])
p = np.array([0.95, 0.30, 0.10, 0.70])          # xác suất vỡ nợ mô hình dự đoán
p_t = np.where(y == 1, p, 1 - p)                # xác suất gán cho NHÃN ĐÚNG
ce = -np.log(p_t)                               # cross-entropy (log loss)

tab = pd.DataFrame({"y": y, "p": p, "p_t": p_t, "CE": ce})
for g in [1, 2]:
    tab[f"(1-p_t)^{g}"] = (1 - p_t) ** g
    tab[f"FL g={g}"] = (1 - p_t) ** g * ce
print(tab.round(3).to_string())


### Ô lệnh 3 -- Vẽ focal loss theo p_t với các giá trị gamma khác nhau

In [ ]:
pt = np.linspace(0.01, 1, 300)
fig, ax = plt.subplots(1, 2, figsize=(8.2, 3))
for g, c in zip([0, 0.5, 1, 2, 5], ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]):
    ax[0].plot(pt, -(1 - pt) ** g * np.log(pt), color=c, label=f"gamma = {g}")
    ax[1].plot(pt, (1 - pt) ** g, color=c)
ax[0].set_ylim(0, 4); ax[0].set_xlabel("p_t"); ax[0].set_ylabel("Mất mát"); ax[0].legend(fontsize=7)
ax[0].set_title("Focal loss (gamma = 0 là cross-entropy)")
ax[1].set_xlabel("p_t"); ax[1].set_ylabel("(1 - p_t)^gamma"); ax[1].set_title("Hệ số giảm trọng số")
plt.savefig("figures/ch06_s03_curves.pdf", bbox_inches="tight")


### Ô lệnh 4 -- Viết focal loss bằng PyTorch và kiểm tra với trường hợp gamma = 0

In [ ]:
def focal_loss(logits, y, gamma=2.0, alpha=None):
    """FL = -alpha_t * (1 - p_t)^gamma * log(p_t), lấy trung bình trên các mẫu."""
    ce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")   # = -log(p_t), ổn định
    p = torch.sigmoid(logits)
    p_t = p * y + (1 - p) * (1 - y)
    loss = (1 - p_t) ** gamma * ce
    if alpha is not None:                       # trọng số lớp: alpha cho lớp 1, 1 - alpha cho lớp 0
        loss = (alpha * y + (1 - alpha) * (1 - y)) * loss
    return loss.mean()

z = torch.tensor([2.0, -0.5, -2.0, 0.8])        # "logit": điểm số trước khi qua sigmoid
yt = torch.tensor([1.0, 1.0, 0.0, 0.0])
print("BCE của PyTorch       :", round(F.binary_cross_entropy_with_logits(z, yt).item(), 4))
print("focal_loss, gamma = 0 :", round(focal_loss(z, yt, gamma=0).item(), 4))
print("focal_loss, gamma = 2 :", round(focal_loss(z, yt, gamma=2).item(), 4))


### Ô lệnh 5 -- Mạng nơ-ron nhiều tầng (MLP) và hàm huấn luyện

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, 64), nn.ReLU(), nn.Dropout(0.1),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
                                 nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)          # trả về logit, mỗi mẫu một số

def train_mlp(X, y, loss_fn, epochs=25, batch=256, lr=1e-3, seed=SEED, verbose=False):
    torch.manual_seed(seed)
    Xt = torch.tensor(np.asarray(X), dtype=torch.float32)
    yt = torch.tensor(np.asarray(y), dtype=torch.float32)
    model = MLP(Xt.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xt))          # xáo trộn dữ liệu mỗi epoch
        for i in range(0, len(Xt), batch):      # chia thành các lô (mini-batch) 256 mẫu
            idx = perm[i:i + batch]
            opt.zero_grad()
            loss = loss_fn(model(Xt[idx]), yt[idx])
            loss.backward()                     # tính đạo hàm
            opt.step()                          # cập nhật trọng số
        if verbose and epoch % 5 == 0:
            print(f"Epoch {epoch:2d}: mất mát của lô cuối = {loss.item():.4f}")
    model.eval()
    return model

@torch.no_grad()
def predict_proba(model, X):
    return torch.sigmoid(model(torch.tensor(np.asarray(X), dtype=torch.float32))).numpy()

bce = lambda z, y: F.binary_cross_entropy_with_logits(z, y)
mlp_bce = train_mlp(Xtr, y_train, bce, verbose=True)


### Ô lệnh 6 -- Đánh giá MLP (hàm mất mát BCE) trên tập test

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             recall_score, precision_score, brier_score_loss)

def report(p, y_true=y_test, t=0.5):
    pred = (p >= t).astype(int)
    return {"ROC-AUC": roc_auc_score(y_true, p), "PR-AUC": average_precision_score(y_true, p),
            "F1": f1_score(y_true, pred), "Recall": recall_score(y_true, pred),
            "Precision": precision_score(y_true, pred), "Brier": brier_score_loss(y_true, p),
            "p TB": p.mean()}

p_bce = predict_proba(mlp_bce, Xte)
print(pd.Series(report(p_bce)).round(3).to_string())


### Ô lệnh 7 -- Nếu xác suất thật là p, focal loss "muốn" dự đoán bao nhiêu?

In [ ]:
q = np.linspace(1e-4, 1 - 1e-4, 200001)
rows = {}
for g in [0, 1, 2, 5]:
    rows[f"gamma = {g}"] = [q[np.argmin(-p_ * (1 - q) ** g * np.log(q) - (1 - p_) * q ** g * np.log(1 - q))]
                            for p_ in [0.05, 0.2, 0.4, 0.6, 0.9]]
print(pd.DataFrame(rows, index=["p = 0,05", "p = 0,2", "p = 0,4", "p = 0,6", "p = 0,9"]).T.round(3))


### Ô lệnh 8 -- So sánh 8 hàm mất mát bằng đánh giá chéo 5-fold (khoảng 2--3 phút)

In [ ]:
from sklearn.model_selection import StratifiedKFold

w_bal = (y_train == 0).sum() / (y_train == 1).sum()
losses = {
    "BCE": bce,
    "BCE có trọng số": lambda z, y: F.binary_cross_entropy_with_logits(z, y, pos_weight=torch.tensor(w_bal)),
    "FL gamma=0.5": lambda z, y: focal_loss(z, y, 0.5),
    "FL gamma=1": lambda z, y: focal_loss(z, y, 1.0),
    "FL gamma=2": lambda z, y: focal_loss(z, y, 2.0),
    "FL gamma=5": lambda z, y: focal_loss(z, y, 5.0),
    "FL gamma=2, alpha=0.25": lambda z, y: focal_loss(z, y, 2.0, 0.25),
    "FL gamma=2, alpha=0.75": lambda z, y: focal_loss(z, y, 2.0, 0.75),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = {}
for name, loss_fn in losses.items():
    folds = []
    for tr, va in skf.split(X_train, y_train):
        qt_f = QuantileTransformer(output_distribution="normal", random_state=0).fit(X_train.iloc[tr])
        m = train_mlp(qt_f.transform(X_train.iloc[tr]), y_train.iloc[tr], loss_fn)
        folds.append(report(predict_proba(m, qt_f.transform(X_train.iloc[va])), y_train.iloc[va]))
    cv_rows[name] = pd.DataFrame(folds).mean()
cv_df = pd.DataFrame(cv_rows).T[["PR-AUC", "F1", "Recall", "Precision", "Brier", "p TB"]]
print(cv_df)


### Ô lệnh 9 -- Phân bố xác suất dự đoán trên tập test

In [ ]:
probs = {"BCE": p_bce}
for name in ["FL gamma=2", "FL gamma=5"]:
    probs[name] = predict_proba(train_mlp(Xtr, y_train, losses[name]), Xte)

fig, ax = plt.subplots(figsize=(5, 2.9))
bins = np.linspace(0, 1, 41)
for (name, p_), c in zip(probs.items(), ["#2a78d6", "#eb6834", "#1baf7a"]):
    ax.hist(p_, bins=bins, histtype="step", lw=1.6, color=c, label=name)
ax.set_xlabel("Xác suất dự đoán (tập test)"); ax.set_ylabel("Số khách hàng"); ax.legend()
plt.savefig("figures/ch06_s09_proba.pdf", bbox_inches="tight")
print({k: round(float(v.mean()), 3) for k, v in probs.items()})


### Ô lệnh 10 -- Công bằng hơn: chọn ngưỡng tốt nhất cho TỪNG hàm mất mát rồi mới so sánh

In [ ]:
from sklearn.model_selection import train_test_split

X_a, X_v, y_a, y_v = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train,
                                      random_state=SEED)       # tách 20% train làm tập kiểm định
qt_a = QuantileTransformer(output_distribution="normal", random_state=0).fit(X_a)
grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
rows = {}
for name in ["BCE", "BCE có trọng số", "FL gamma=2", "FL gamma=2, alpha=0.75"]:
    m = train_mlp(qt_a.transform(X_a), y_a, losses[name])
    p_v = predict_proba(m, qt_a.transform(X_v))
    t_best = grid[np.argmax([f1_score(y_v, (p_v >= t).astype(int)) for t in grid])]
    p_te = predict_proba(m, qt_a.transform(X_test))
    pred = (p_te >= t_best).astype(int)
    rows[name] = {"Ngưỡng t*": t_best, "F1 test": f1_score(y_test, pred),
                  "Recall": recall_score(y_test, pred), "PR-AUC": average_precision_score(y_test, p_te)}
print(pd.DataFrame(rows).T.round(3))
